In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"
os.environ["MUJOCO_GL"] = "glfw"

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot training
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.optim.optimizers import AdamWConfig
from lerobot.configs.default import WandBConfig
from lerobot.utils.utils import init_logging
from lerobot.scripts.train import train

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint

# my code
from scripts.utils.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)


C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Must be run as admin!

In [2]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
PUSH_TO_HUB = False
REPO_NAME = 'so101_table_leg_move'
EXPERIMENT_NAME = '50_episodes_no_augmentations'

In [ ]:
DATASET_PATH = DATASETS_DIR / REPO_NAME
DATASET_ID = f"{HF_NAME}/{REPO_NAME}"

In [4]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

In [5]:
wandb.login(key = os.getenv('WANDB_API_KEY'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\jonathan\_netrc
wandb: Currently logged in as: jonathm126 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)

In [7]:
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"Robot type: {ds_meta.robot_type}")
print(f"keys to access images from cameras: {ds_meta.camera_keys=}\n")

Total number of episodes: 50
Average number of frames per episode: 534.620
Frames per second used during data collection: 30
Robot type: so101_follower
keys to access images from cameras: ds_meta.camera_keys=['observation.images.wrist_cam', 'observation.images.top_cam']



In [8]:
print("Features:")
pprint(ds_meta.features.keys())

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])


In [9]:
# print('observation.images.wrist_cam')
# pprint(ds_meta.features['observation.images.wrist_cam'])

# print('observation.images.top_cam')
# pprint(ds_meta.features['observation.images.top_cam'])

print('observation.state')
pprint(ds_meta.features['observation.state'])

observation.state
{'dtype': 'float32',
 'names': ['shoulder_pan.pos',
           'shoulder_lift.pos',
           'elbow_flex.pos',
           'wrist_flex.pos',
           'wrist_roll.pos',
           'gripper.pos'],
 'shape': (6,)}


In [10]:
print('Action Space:')
pprint(ds_meta.features['action'])

Action Space:
{'dtype': 'float32',
 'names': ['shoulder_pan.pos',
           'shoulder_lift.pos',
           'elbow_flex.pos',
           'wrist_flex.pos',
           'wrist_roll.pos',
           'gripper.pos'],
 'shape': (6,)}


In [11]:
# select image transforms
transforms_cfg = ImageTransformsConfig(enable = False)

# select episodes
episodes = list(range(len(ds_meta.episodes)))

# build dataset
dataset_cfg = DatasetConfig(repo_id = DATASET_ID,
                            root               = DATASET_PATH,   # dataset location
                            episodes           = episodes,       # which episodes to use
                            image_transforms   = transforms_cfg, # configure image transformations
                            use_imagenet_stats = True,           # normalize image using imagenet weights
                            video_backend      = 'pyav'          # for windows
                            )

Policy Config

In [12]:
input_features = {
    "observation.images.wrist_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.images.top_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.state": {
        "type" : "STATE",
        "shape": [6]
    }
}

output_features = {
    "action": {
        "type" : "ACTION",
        "shape": [6]
    }
}

In [ ]:
policy_config = ACTConfig(n_obs_steps = 1, # how many observations does the policy need
                            chunk_size      = 100,            # action chunking size
                            n_action_steps  = 100,             # how many actions to preform (?) TODO - reduce to 10
                            device          = device,
                            input_features  = input_features,
                            output_features = output_features,
                            push_to_hub     = False,
                            )

Training params config:

In [14]:
optimizer_cfg = AdamWConfig(lr = 1e-5, weight_decay = 1e-4)

WandB logging:

In [15]:
wandb_config = WandBConfig(enable = True, 
                            disable_artifact = False,
                            project          = 'ACT-Real-Sim-Real',
                            entity           = 'jonathm126-personal',
                            mode             = 'online',
                            run_id           =  EXPERIMENT_NAME + '-train' ) # careful - this is a unique ids
print(wandb_config.run_id)

so101_table_leg_move-50_episodes_no_augmentations-train


Integration:

In [ ]:
train_cfg = TrainPipelineConfig(dataset                    = dataset_cfg,
                                # env                        = env_config,                 # sim env
                                policy                     = policy_config,              # policy config
                                output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
                                job_name                   = REPO_NAME + '-' + EXPERIMENT_NAME + '-train', # name
                                resume                     = False,                      # only if resuming!!
                                use_policy_training_preset = True,
                                optimizer                  = optimizer_cfg,
                                wandb                      = wandb_config,
                                steps                      = 100000,
                                log_freq                   = 200,
                                # eval_freq                  = 20000,
                                save_checkpoint            = True,
                                save_freq                  = 20000,
                                )

Train:

In [23]:
init_logging()
train(train_cfg)

INFO 2025-07-13 23:53:08 ts\train.py:111 {'batch_size': 8,
 'dataset': {'episodes': [0,
                          1,
                          2,
                          3,
                          4,
                          5,
                          6,
                          7,
                          8,
                          9,
                          10,
                          11,
                          12,
                          13,
                          14,
                          15,
                          16,
                          17,
                          18,
                          19,
                          20,
                          21,
                          22,
                          23,
                          24,
                          25,
                          26,
                          27,
                          28,
                          29,
                          30,
                     

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
INFO 2025-07-13 23:53:12 db_utils.py:103 Track this run --> https://wandb.ai/jonathm126-personal/ACT-Real-Sim-Real/runs/so101_table_leg_move-50_episodes_no_augmentations-train
INFO 2025-07-13 23:53:12 ts\train.py:127 Creating dataset


Logs will be synced with wandb.


Generating train split: 25013 examples [00:00, 149010.42 examples/s]
INFO 2025-07-13 23:53:14 ts\train.py:138 Creating policy
INFO 2025-07-13 23:53:15 ts\train.py:144 Creating optimizer and scheduler
INFO 2025-07-13 23:53:15 ts\train.py:156 Output dir: C:\Github\IDC_Project-Sim_Real_Sim\policies\so101_table_leg_move-50_episodes_no_augmentations
INFO 2025-07-13 23:53:15 ts\train.py:159 cfg.steps=100000 (100K)
INFO 2025-07-13 23:53:15 ts\train.py:160 dataset.num_frames=25013 (25K)
INFO 2025-07-13 23:53:15 ts\train.py:161 dataset.num_episodes=50
INFO 2025-07-13 23:53:15 ts\train.py:162 num_learnable_params=51597190 (52M)
INFO 2025-07-13 23:53:15 ts\train.py:163 num_total_params=51597238 (52M)
INFO 2025-07-13 23:53:15 ts\train.py:202 Start offline training on a fixed dataset
INFO 2025-07-13 23:54:57 ts\train.py:232 step:200 smpl:2K ep:3 epch:0.06 loss:7.046 grdn:157.003 lr:1.0e-05 updt_s:0.413 data_s:0.096
INFO 2025-07-13 23:56:17 ts\train.py:232 step:400 smpl:3K ep:6 epch:0.13 loss:3.040 

KeyboardInterrupt: 